# Otimizadores

O material anterior mostrou de onde vêm os gradientes. Este trata do que fazer com eles. O otimizador recebe os parâmetros e seus gradientes e decide o passo de atualização,

$$
\theta_{t+1} = \theta_t - \Delta_t
$$

em que $\theta_t$ são os parâmetros na iteração $t$ e $\Delta_t$ é o passo. Tudo o que distingue um otimizador de outro está em como $\Delta_t$ é calculado a partir do histórico de gradientes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Gradiente descendente

O passo mais simples é o próprio gradiente, multiplicado pela taxa de aprendizado,

$$
\Delta_t = \eta \, \nabla L(\theta_t)
$$

em que $\eta$ é a taxa de aprendizado e $\nabla L(\theta_t)$ é o gradiente do custo. O passo tem o mesmo tamanho relativo em todas as direções, o que é ruim quando o custo desce muito mais rápido em uma direção do que em outra. Nesses vales alongados a trajetória oscila entre as paredes e avança pouco no fundo.

## Momento

O momento guarda uma média das direções anteriores e caminha por ela, em vez de reagir só ao gradiente atual,

$$
v_{t+1} = \beta v_t + \nabla L(\theta_t)
\qquad
\theta_{t+1} = \theta_t - \eta \, v_{t+1}
$$

em que $v_t$ é a velocidade acumulada e $\beta$ é o coeficiente de momento, tipicamente 0.9. As oscilações de sinal alternado se cancelam na média, e as direções consistentes se somam, o que acelera a descida em vales alongados.

## AdaGrad

Os métodos adaptativos dão um passo diferente para cada parâmetro. O AdaGrad divide o passo pela raiz da soma acumulada dos quadrados dos gradientes daquele parâmetro,

$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t} + \epsilon} \, \nabla L(\theta_t)
$$

em que $G_t$ é essa soma acumulada e $\epsilon$ é um valor pequeno que evita divisão por zero. Parâmetros que recebem gradientes grandes passam a andar menos, o que ajuda em problemas esparsos. O defeito é que $G_t$ só cresce, então a taxa efetiva encolhe até o treinamento parar de avançar.

## RMSprop

O RMSprop troca a soma acumulada por uma média móvel exponencial, que esquece o passado distante,

$$
v_t = \beta v_{t-1} + (1 - \beta) \left(\nabla L(\theta_t)\right)^2
\qquad
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t} + \epsilon} \, \nabla L(\theta_t)
$$

em que $\beta$ é o fator de decaimento, tipicamente 0.9. Como $v_t$ pode diminuir, a taxa efetiva volta a crescer quando os gradientes ficam pequenos, e o método continua avançando.

## Adam

O Adam combina as duas ideias, guardando uma média móvel dos gradientes e outra dos seus quadrados,

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(\theta_t)
\qquad
v_t = \beta_2 v_{t-1} + (1 - \beta_2) \left(\nabla L(\theta_t)\right)^2 .
$$

As duas médias começam em zero e por isso ficam enviesadas para baixo nas primeiras iterações, o que se corrige dividindo pelo fator que falta,

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}
\qquad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
\qquad
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \, \hat{m}_t .
$$

Os valores usuais são $\beta_1 = 0.9$ e $\beta_2 = 0.999$. O numerador traz o momento e o denominador traz a adaptação por parâmetro, e é essa combinação que faz do Adam a escolha padrão em redes profundas.

## A função de Himmelblau

Para ver as trajetórias, é melhor um problema de dois parâmetros, que cabe em um gráfico. A função de Himmelblau,

$$
f(x_1, x_2) = (x_1^2 + x_2 - 11)^2 + (x_1 + x_2^2 - 7)^2
$$

tem quatro mínimos, todos com valor zero, separados por cristas e pontos de sela. É um relevo difícil o bastante para separar os métodos.

In [ ]:
def himmelblau(x):
    x1, x2 = x[0], x[1]
    return (x1 ** 2 + x2 - 11) ** 2 + (x1 + x2 ** 2 - 7) ** 2


minima = torch.tensor([[3.0, 2.0], [-2.805, 3.131], [-3.779, -3.283], [3.584, -1.848]])

In [ ]:
grid_x1, grid_x2 = np.meshgrid(np.linspace(-6, 6, 400), np.linspace(-6, 6, 400))
grid_z = himmelblau([grid_x1, grid_x2])

plt.figure(figsize=(8, 6))
plt.contour(grid_x1, grid_x2, grid_z, levels=np.logspace(0, 5, 35), cmap="viridis")
plt.scatter(minima[:, 0], minima[:, 1], color="r", marker="*", s=200, label="mínimos")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

Cada otimizador parte do mesmo ponto, com a mesma taxa de aprendizado e o mesmo número de iterações, de modo que a única diferença entre as trajetórias é a regra de atualização. O tensor de dois valores faz o papel dos parâmetros do modelo.

In [ ]:
def run_optimizer(optimizer_class, start, steps=100, learning_rate=0.03, **kwargs):
    x = start.clone().detach().requires_grad_(True)
    optimizer = optimizer_class([x], lr=learning_rate, **kwargs)

    track = []
    losses = []
    for _ in range(steps):
        loss = himmelblau(x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        track.append(x.detach().clone())
        losses.append(loss.item())

    return torch.stack(track), losses

In [ ]:
start = torch.tensor([-4.0, 0.0])

configurations = {
    "SGD": (torch.optim.SGD, {}),
    "SGD com momento": (torch.optim.SGD, {"momentum": 0.2}),
    "AdaGrad": (torch.optim.Adagrad, {}),
    "RMSprop": (torch.optim.RMSprop, {"alpha": 0.9}),
    "Adam": (torch.optim.Adam, {}),
}

tracks = {}
curves = {}
for name, (optimizer_class, kwargs) in configurations.items():
    tracks[name], curves[name] = run_optimizer(optimizer_class, start, **kwargs)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.contour(grid_x1, grid_x2, grid_z, levels=np.logspace(0, 5, 35), cmap="Greys", alpha=0.5)
for name, track in tracks.items():
    ax1.plot(track[:, 0], track[:, 1], marker=".", label=name)
ax1.scatter(start[0], start[1], color="k", marker="*", s=200)
ax1.set_xlim(-6, 6)
ax1.set_ylim(-6, 6)
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")
ax1.legend()

for name, losses in curves.items():
    ax2.plot(losses, label=name)
ax2.set_yscale("log")
ax2.set_xlabel("iteração")
ax2.set_ylabel("f(x1, x2)")
ax2.legend()
ax2.grid(True)
plt.show()

As trajetórias partem do mesmo ponto e seguem para bacias diferentes, o que já mostra que a regra de atualização decide para onde o método vai, e não apenas quão rápido chega. O SGD e o momento descem para o mínimo próximo de $(-2.8, 3.1)$, enquanto o RMSprop e o Adam terminam perto de $(-3.8, -3.3)$.

Em cem iterações, o RMSprop e o Adam já estão na vizinhança de um mínimo, o SGD ainda está a caminho, e o AdaGrad é o que menos avança. Esse último resultado é exatamente o defeito descrito na seção dele: o denominador acumula os quadrados de todos os gradientes vistos até ali, só cresce, e a taxa efetiva encolhe até o método quase parar.

Vale reparar que todos usaram a mesma taxa de aprendizado, o que não favorece nenhum deles em particular. Comparar otimizadores sem ajustar a taxa de cada um mede tanto a regra de atualização quanto a sorte de a taxa escolhida servir para ela.

## Comparação no MNIST

Em uma rede de verdade não há trajetória para desenhar, e a comparação passa a ser entre as curvas de perda e de acurácia. Para o treinamento ficar rápido, apenas uma parte do MNIST é usada.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(3000))
validation_set = Subset(full_test_set, range(1000))

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)

print(f"treino: {len(train_set)}, validação: {len(validation_set)}")

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.layers(x)   # [batch, 10]

A função de treinamento é a mesma do material anterior, com o otimizador entrando como argumento. Cada otimizador recebe um modelo recém-criado, com a mesma semente, para que todos partam dos mesmos pesos iniciais.

In [ ]:
criterion = nn.CrossEntropyLoss()


def accuracy(model, dataloader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()

    return correct / len(dataloader.dataset)

In [ ]:
def train(model, optimizer, epochs=10):
    history = {"loss": [], "accuracy": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        history["loss"].append(running_loss / len(train_set))
        history["accuracy"].append(accuracy(model, validation_dataloader))

    return history

In [ ]:
learning_rate = 0.01
results = {}

for name, (optimizer_class, kwargs) in configurations.items():
    torch.manual_seed(42)
    model = MLP().to(device)
    optimizer = optimizer_class(model.parameters(), lr=learning_rate, **kwargs)

    results[name] = train(model, optimizer)
    print(f"{name}: perda final {results[name]['loss'][-1]:.4f}, "
          f"acurácia final {results[name]['accuracy'][-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, history in results.items():
    ax1.plot(history["loss"], label=name)
    ax2.plot(history["accuracy"], label=name)

ax1.set_xlabel("época")
ax1.set_ylabel("entropia cruzada de treino")
ax1.legend()
ax1.grid(True)

ax2.set_xlabel("época")
ax2.set_ylabel("acurácia de validação")
ax2.legend()
ax2.grid(True)
plt.show()

Com poucos dados e poucas épocas, a diferença entre os métodos aparece principalmente na velocidade. O SGD puro é o mais lento, porque nada corrige a escala do passo. O coeficiente de momento usado aqui é 0.2, um valor conservador, e o ganho que ele traz é pequeno na mesma proporção, o que o Exercício 3 explora. Os métodos adaptativos descem bem mais rápido, e o AdaGrad, que travou na função de Himmelblau, vai bem aqui, porque dez épocas não são suficientes para o denominador crescer a ponto de atrapalhar.

Perda de treino menor não significa modelo melhor. A acurácia de validação é que diz se o ganho vale alguma coisa, e a distância entre as duas curvas é o assunto do material sobre regularização.

## Exercícios

### Exercício 1

Repita a comparação na função de Himmelblau partindo de outros pontos iniciais, como `[0.0, 0.0]` e `[4.0, 4.0]`. Algum otimizador chega sempre ao mesmo mínimo, independentemente de onde começa?

In [ ]:
start = torch.tensor([0.0, 0.0])

### Exercício 2

Escolha dois otimizadores e treine cada um com três taxas de aprendizado diferentes no MNIST. Qual dos dois é mais sensível à escolha da taxa?

In [ ]:
learning_rates = [0.001, 0.01, 0.1]

### Exercício 3

O momento do `torch.optim.SGD` é controlado pelo argumento `momentum`. Treine com 0.0, 0.5 e 0.9 e compare as curvas de perda. O que acontece quando o valor se aproxima de 1?

In [ ]:
momentums = [0.0, 0.5, 0.9]